# HAL API Enrichment Pipeline

Matches documents from `almanach_documents_df.parquet` to HAL API records from the ALMAnaCH team,
extracts rich metadata (authors, institutions, keywords, domains), and saves an enriched parquet
containing **only matched documents**.

In [ ]:
import pandas as pd
import numpy as np
import requests
from collections import Counter

df = pd.read_parquet('/data/gbg141/Arlequin/data/almanach_documents_df.parquet')
print(f"Loaded {len(df)} documents")

## Step 1: Fetch all ALMAnaCH papers from HAL API

In [ ]:
HAL_API_URL = "https://api.archives-ouvertes.fr/search/"
ALMANACH_STRUCT_ID = 482775

HAL_FIELDS = [
    "title_s", "halId_s", "abstract_s",
    "authFullName_s",
    "structId_i", "structName_s", "structType_s", "labStructId_i",
    "keyword_s", "domain_s", "producedDateY_i",
]

resp = requests.get(HAL_API_URL, params={
    "q": f"structId_i:{ALMANACH_STRUCT_ID}",
    "fl": ",".join(HAL_FIELDS),
    "rows": 1000,
    "wt": "json",
})
resp.raise_for_status()
hal_docs = resp.json()["response"]["docs"]
print(f"Fetched {len(hal_docs)} ALMAnaCH papers from HAL")

with_abstract = sum(1 for d in hal_docs if d.get("abstract_s"))
with_authors = sum(1 for d in hal_docs if d.get("authFullName_s"))
print(f"  With abstracts: {with_abstract}")
print(f"  With authors:   {with_authors}")

## Step 2: Multi-pass matching

In [ ]:
matched = {}  # doc_idx -> (hal_doc, match_method)

# Pass 1: Title-in-body (case-insensitive)
for hdoc in hal_docs:
    if "title_s" not in hdoc:
        continue
    title = hdoc["title_s"][0].lower().strip()
    if len(title) < 20:
        continue
    for idx in range(len(df)):
        if idx in matched:
            continue
        body = df["body_text"].iloc[idx].lower()
        if title in body:
            matched[idx] = (hdoc, "title_in_body")
            break

print(f"Pass 1 (title-in-body): {len(matched)} matched")

In [ ]:
# Pass 2: Abstract first sentence in body
before = len(matched)
for hdoc in hal_docs:
    if not hdoc.get("abstract_s"):
        continue
    abstract = hdoc["abstract_s"][0]
    sents = [s.strip() for s in abstract.split(".") if len(s.strip()) > 40]
    if not sents:
        continue
    search = sents[0][:100].lower()
    for idx in range(len(df)):
        if idx in matched:
            continue
        body = df["body_text"].iloc[idx].lower()
        if search in body:
            matched[idx] = (hdoc, "abstract_in_body")
            break

print(f"Pass 2 (abstract-in-body): +{len(matched) - before} = {len(matched)} total")

In [ ]:
# Pass 3: Abstract mid-chunk in body
before = len(matched)
for hdoc in hal_docs:
    if not hdoc.get("abstract_s"):
        continue
    abstract = hdoc["abstract_s"][0].lower()
    if len(abstract) > 80:
        chunk = abstract[20:80]
        for idx in range(len(df)):
            if idx in matched:
                continue
            body = df["body_text"].iloc[idx].lower()
            if chunk in body:
                matched[idx] = (hdoc, "abstract_midchunk")
                break

print(f"Pass 3 (abstract mid-chunk): +{len(matched) - before} = {len(matched)} total")

In [ ]:
# Pass 4: Author surname matching (all last names must appear in body)
before = len(matched)
for hdoc in hal_docs:
    if not hdoc.get("authFullName_s"):
        continue
    authors = hdoc["authFullName_s"]
    if len(authors) < 2:
        continue
    lastnames = [a.split()[-1].lower() for a in authors[:4]]
    for idx in range(len(df)):
        if idx in matched:
            continue
        body = df["body_text"].iloc[idx].lower()
        if all(ln in body for ln in lastnames):
            matched[idx] = (hdoc, "author_surnames")
            break

print(f"Pass 4 (author surnames): +{len(matched) - before} = {len(matched)} total")

In [ ]:
# Summary
print(f"\n=== Matching Summary ===")
print(f"Total documents:  {len(df)}")
print(f"Matched:          {len(matched)}")
print(f"Unmatched:        {len(df) - len(matched)}")
print()
method_counts = Counter(m for _, m in matched.values())
for method, count in method_counts.most_common():
    print(f"  {method}: {count}")
print()

print("Sample matches:")
for idx in sorted(matched.keys())[:5]:
    hdoc, method = matched[idx]
    title = hdoc.get("title_s", ["?"])[0][:70]
    authors = hdoc.get("authFullName_s", [])[:3]
    print(f"  Doc {idx} [{method}]: {hdoc['halId_s']} - {title}")
    print(f"    Authors: {authors}")

## Step 3: Build enriched DataFrame (matched docs only)

In [ ]:
rows = []
for idx in sorted(matched.keys()):
    hdoc, method = matched[idx]
    rows.append({
        "body_text": df["body_text"].iloc[idx],
        "embedding": df["embedding"].iloc[idx],
        "hal_id": hdoc.get("halId_s", ""),
        "title": hdoc.get("title_s", [""])[0],
        "authors": hdoc.get("authFullName_s", []),
        "struct_names": hdoc.get("structName_s", []),
        "struct_ids": [int(x) for x in hdoc.get("structId_i", [])],
        "struct_types": hdoc.get("structType_s", []),
        "keywords": hdoc.get("keyword_s", []),
        "domains": hdoc.get("domain_s", []),
        "year": hdoc.get("producedDateY_i", None),
        "match_method": method,
    })

df_enriched = pd.DataFrame(rows)
print(f"Enriched DataFrame: {df_enriched.shape}")
print(f"Columns: {df_enriched.columns.tolist()}")
df_enriched[["hal_id", "title", "match_method"]].head(10)

## Step 4: Inspect enriched metadata

In [ ]:
# Author statistics
all_authors = [a for authors in df_enriched["authors"] for a in authors]
author_counts = Counter(all_authors)
print(f"Unique authors: {len(author_counts)}")
print(f"\nTop 15 most prolific authors:")
for author, count in author_counts.most_common(15):
    print(f"  {count:3d} papers - {author}")

authors_per_paper = df_enriched["authors"].apply(len)
print(f"\nAuthors per paper: min={authors_per_paper.min()}, "
      f"max={authors_per_paper.max()}, mean={authors_per_paper.mean():.1f}")

In [ ]:
# Institution statistics
all_structs = []
for names, types in zip(df_enriched["struct_names"], df_enriched["struct_types"]):
    for n, t in zip(names, types):
        all_structs.append((n, t))

struct_counts = Counter(all_structs)
print("Top 20 structures (name, type):")
for (name, stype), count in struct_counts.most_common(20):
    print(f"  {count:3d} papers - [{stype}] {name}")

In [ ]:
# Domain statistics
all_domains = [d for domains in df_enriched["domains"] for d in domains]
domain_counts = Counter(all_domains)
print("Top 15 domains:")
for domain, count in domain_counts.most_common(15):
    print(f"  {count:3d} - {domain}")

In [ ]:
# Year distribution
print("Papers per year:")
print(df_enriched["year"].value_counts().sort_index())

## Step 5: Save enriched parquet

In [ ]:
output_path = "/data/gbg141/Arlequin/data/almanach_documents_enriched.parquet"
df_enriched.to_parquet(output_path, index=False)
print(f"Saved enriched parquet ({len(df_enriched)} docs) to {output_path}")

# Verify
df_check = pd.read_parquet(output_path)
print(f"Verification: {df_check.shape}")
print(f"Columns: {df_check.columns.tolist()}")